In [ ]:
#!/usr/bin/env python3
"""
Rival ansatz sweep for alpha^-1 numerical coincidence.
Author: Edward P. López (with computational assistance)
Date: 2026-09-16
License: MIT
"""

import numpy as np
import itertools
from dataclasses import dataclass

# --- Target ---
ALPHA_INV_EXP = 137.035999084  # CODATA 2018
# (si prefieres CODATA 2022: 137.035999177)

# --- Thresholds (declared a priori) ---
EPS_STRICT = 1e-6   # primary threshold
EPS_LOOSE  = 1e-5   # secondary threshold
EPS_SCAN   = 1e-3   # for listing candidates

# --- Search bound ---
K = 10  # |coefficient| <= K

# --- Constants ---
PI = np.pi
E  = np.e
S2 = np.sqrt(2)
PHI = (1 + np.sqrt(5)) / 2

@dataclass
class Hit:
    coeffs: tuple
    value: float
    err_abs: float
    err_rel: float

def sweep(name, basis, K=K):
    """
    basis: list of tuples (label, constant_value).
    Enumerates a_1*c_1 + a_2*c_2 + ... with |a_i| <= K.
    Returns list of hits with err_rel < EPS_SCAN, sorted by err_rel.
    """
    n = len(basis)
    values = [c for _, c in basis]
    labels = [l for l, _ in basis]

    hits = []
    total = (2*K + 1)**n

    for coeffs in itertools.product(range(-K, K+1), repeat=n):
        v = sum(a * c for a, c in zip(coeffs, values))
        err_abs = abs(v - ALPHA_INV_EXP)
        err_rel = err_abs / ALPHA_INV_EXP
        if err_rel < EPS_SCAN:
            hits.append(Hit(coeffs, v, err_abs, err_rel))

    hits.sort(key=lambda h: h.err_rel)

    # Summary
    n_strict = sum(1 for h in hits if h.err_rel < EPS_STRICT)
    n_loose  = sum(1 for h in hits if h.err_rel < EPS_LOOSE)

    print(f"\n{'='*70}")
    print(f"SPACE: {name}")
    print(f"Basis: {labels}")
    print(f"Total combinations: {total}")
    print(f"Hits with eps_rel < {EPS_STRICT:.0e}: {n_strict}")
    print(f"Hits with eps_rel < {EPS_LOOSE:.0e}: {n_loose}")
    print(f"Hits with eps_rel < {EPS_SCAN:.0e}: {len(hits)}")

    if hits:
        print(f"\nTop 5 candidates:")
        print(f"{'coeffs':<30} {'value':>15} {'err_abs':>12} {'err_rel':>12}")
        for h in hits[:5]:
            print(f"{str(h.coeffs):<30} {h.value:>15.9f} {h.err_abs:>12.3e} {h.err_rel:>12.3e}")

        # Gap analysis
        if len(hits) >= 2:
            gap = hits[1].err_rel / hits[0].err_rel
            print(f"\nGap (2nd best / best): {gap:.2f}x")
    else:
        print("No hits within scan threshold.")

    return hits

# --- The five spaces ---
spaces = {
    "S1: a*pi^3 + b*pi^2 + c*pi": [
        ("pi^3", PI**3),
        ("pi^2", PI**2),
        ("pi",   PI),
    ],
    "S2: a*pi^3 + b*pi^2 + c*e + d": [
        ("pi^3", PI**3),
        ("pi^2", PI**2),
        ("e",    E),
        ("1",    1.0),
    ],
    "S3: a*e^3 + b*e^2 + c*e": [
        ("e^3", E**3),
        ("e^2", E**2),
        ("e",   E),
    ],
    "S4: a*pi^3 + b*pi^2 + c*sqrt(2) + d*phi": [
        ("pi^3",  PI**3),
        ("pi^2",  PI**2),
        ("sqrt2", S2),
        ("phi",   PHI),
    ],
    "S5: a*pi^4 + b*pi^3 + c*pi^2 + d*pi": [
        ("pi^4", PI**4),
        ("pi^3", PI**3),
        ("pi^2", PI**2),
        ("pi",   PI),
    ],
}

if __name__ == "__main__":
    print(f"Target: alpha^-1 = {ALPHA_INV_EXP}")
    print(f"Bound:  |a_i| <= {K}")

    all_results = {}
    for name, basis in spaces.items():
        all_results[name] = sweep(name, basis)

    print(f"\n{'='*70}")
    print("SUMMARY TABLE")
    print(f"{'Space':<45} {'<1e-6':>8} {'<1e-5':>8} {'best err_rel':>15}")
    for name, hits in all_results.items():
        n_strict = sum(1 for h in hits if h.err_rel < EPS_STRICT)
        n_loose  = sum(1 for h in hits if h.err_rel < EPS_LOOSE)
        best = hits[0].err_rel if hits else float('nan')
        print(f"{name:<45} {n_strict:>8} {n_loose:>8} {best:>15.3e}")

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# Recolectar valores de S1
pi = np.pi
target = 137.035999084
K = 10
values = []
for a in range(-K, K+1):
    for b in range(-K, K+1):
        for c in range(-K, K+1):
            values.append(a*pi**3 + b*pi**2 + c*pi)
values = np.array(values)

# Histograma sobre el rango efectivo
lo, hi = np.percentile(values, [1, 99])
hist, edges = np.histogram(values, bins=50, range=(lo, hi))
edges_mid = (edges[:-1] + edges[1:]) / 2

# Comparación con uniforme
expected = len(values[(values >= lo) & (values <= hi)]) / 50
chi2 = np.sum((hist - expected)**2 / expected)

print(f"Chi^2 = {chi2:.2f} with 49 dof")
print(f"Expected chi^2 ~ 49 ± 9.9")
print(f"Ratio: {chi2/49:.3f}")

In [ ]:

#!/usr/bin/env python3
"""
Monte Carlo null distribution for the S1 hit at 10^-5.
Empirical replacement for the failed equidistribution/Poisson test.
"""

import numpy as np

PI = np.pi
K = 10
EPS = 1e-5
N_TRIALS = 10000
TARGET_LO = 100.0
TARGET_HI = 200.0

# Precompute all S1 values once
values = []
coeffs = []
for a in range(-K, K+1):
    for b in range(-K, K+1):
        for c in range(-K, K+1):
            values.append(a*PI**3 + b*PI**2 + c*PI)
            coeffs.append((a, b, c))
values = np.array(values)

# Empirical distribution of hits at random targets
rng = np.random.default_rng(42)
hit_counts = []

for _ in range(N_TRIALS):
    target = rng.uniform(TARGET_LO, TARGET_HI)
    err_rel = np.abs(values - target) / target
    n_hits = np.sum(err_rel < EPS)
    hit_counts.append(n_hits)

hit_counts = np.array(hit_counts)

# Empirical p-value: fraction of trials with >= 1 hit
p_empirical = np.mean(hit_counts >= 1)
# Fraction with exactly 1 hit (what we observe in S1)
p_exactly_1 = np.mean(hit_counts == 1)
# Mean and 95th percentile of hit count
mean_hits = hit_counts.mean()
q95 = np.percentile(hit_counts, 95)

print(f"Number of trials: {N_TRIALS}")
print(f"Range of targets: [{TARGET_LO}, {TARGET_HI}]")
print(f"Tolerance: {EPS}")
print()
print(f"Empirical P(>=1 hit) = {p_empirical:.4f}")
print(f"Empirical P(exactly 1 hit) = {p_exactly_1:.4f}")
print(f"Mean hit count per trial = {mean_hits:.4f}")
print(f"95th percentile = {q95}")
print()
print(f"Observed in S1 against alpha^-1_exp = 1 hit")
print(f"Empirical p-value = {p_empirical:.4f} ({p_empirical*100:.2f}%)")
print()

# Also report local density near 137
window = 0.1  # narrow window around 137
local = values[(values > 137 - window) & (values < 137 + window)]
expected_local = 2 * window * len(values) / (TARGET_HI - TARGET_LO)
print(f"Local count in [136.9, 137.1]: {len(local)} values")
print(f"Expected if uniform: {expected_local:.1f}")
print(f"Local enhancement factor: {len(local)/expected_local:.2f}x")

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
pi = np.pi
K = 10
values = [a*pi**3 + b*pi**2 + c*pi
          for a in range(-K,K+1)
          for b in range(-K,K+1)
          for c in range(-K,K+1)]
plt.hist(values, bins=100, range=(-440,440))
plt.axvline(137.036, color='red', linestyle='--', label=r'$\alpha^{-1}_{exp}$')
plt.xlabel(r'$a\pi^3 + b\pi^2 + c\pi$')
plt.ylabel('Count')
plt.legend()
plt.title('S1 distribution and the target value')
plt.savefig('fig1_s1_histogram.png', dpi=300)

In [ ]:

expected_local = 2 * window * len(values) / (TARGET_HI - TARGET_LO)
# = 0.2 * 9261 / (200 - 100) = 18.5

In [ ]:
plt.hist(values, bins=100, range=(-440,440), label='S1 values')
plt.axvline(137.036, color='red', linestyle='--',
            label=r'$\alpha^{-1}_{exp} = 137.036$')
plt.axvline(136.591, color='orange', linestyle=':',
            label='(4,0,4) = 136.591')  # segundo candidato empírico más cercano
plt.legend()

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

pi = np.pi
K = 10
values = [a*pi**3 + b*pi**2 + c*pi
          for a in range(-K,K+1)
          for b in range(-K,K+1)
          for c in range(-K,K+1)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Panel izquierdo: distribución global
ax1.hist(values, bins=100, range=(-440,440), label='S1 values')
ax1.axvline(137.036, color='red', linestyle='--',
            label=r'$\alpha^{-1}_{exp} = 137.036$')
ax1.set_xlabel(r'$a\pi^3 + b\pi^2 + c\pi$')
ax1.set_ylabel('Count')
ax1.set_title('(a) Global distribution')
ax1.legend()

# Panel derecho: zoom en la región del hit
zoom_lo, zoom_hi = 136.5, 137.1
ax2.hist(values, bins=50, range=(zoom_lo, zoom_hi),
         color='steelblue', label='S1 values')
ax2.axvline(137.036, color='red', linestyle='--', linewidth=2,
            label=r'$\alpha^{-1}_{exp} = 137.036$')
ax2.axvline(136.591, color='orange', linestyle=':', linewidth=2,
            label=r'$(4,0,4) = 136.591$')
ax2.axvline(137.0363, color='green', linestyle='-', linewidth=2,
            label=r'$(4,1,1) = 137.0363$')
ax2.set_xlabel(r'$a\pi^3 + b\pi^2 + c\pi$')
ax2.set_ylabel('Count')
ax2.set_title('(b) Zoom: region around the target')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('fig1_s1_distribution.png', dpi=300)

In [ ]:

#!/usr/bin/env python3
"""
Figure 1 for the paper on the pi^3 ansatz for alpha^-1.
Two-panel figure: global distribution + zoom at the hit.
"""

import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------
PI = np.pi
K = 10
TARGET = 137.035999084      # CODATA 2018
HIT_411 = 4*PI**3 + PI**2 + PI       # (4,1,1) — the hit
SECOND_BEST = 3*PI**3 + 7*PI**2 - 8*PI   # (3,7,-8) — second best
GAP_FACTOR = 205.7  # computed from delta_2/delta_1

# ---------------------------------------------------------------
# Generate S1 values
# ---------------------------------------------------------------
values = []
for a in range(-K, K+1):
    for b in range(-K, K+1):
        for c in range(-K, K+1):
            values.append(a*PI**3 + b*PI**2 + c*PI)
values = np.array(values)

# ---------------------------------------------------------------
# Figure: two panels
# ---------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- Panel (a): global distribution ---
ax1.hist(values, bins=100, range=(-440, 440),
         color='steelblue', edgecolor='none', label='S1 values')
ax1.axvline(TARGET, color='red', linestyle='--', linewidth=1.5,
            label=r'$\alpha^{-1}_{exp} = 137.036$')
ax1.set_xlabel(r'$a\pi^3 + b\pi^2 + c\pi$', fontsize=11)
ax1.set_ylabel('Count', fontsize=11)
ax1.set_title('(a) Global distribution', fontsize=12)
ax1.legend(fontsize=10, loc='upper left')
ax1.grid(alpha=0.2)

# --- Panel (b): zoom around the target ---
zoom_lo, zoom_hi = 136.9, 137.1
ax2.hist(values, bins=100, range=(zoom_lo, zoom_hi),
         color='steelblue', edgecolor='none', label='S1 values')

# Target line
ax2.axvline(TARGET, color='red', linestyle='--', linewidth=1.8,
            label=r'$\alpha^{-1}_{exp} = 137.036$')

# The hit (4,1,1)
ax2.axvline(HIT_411, color='green', linestyle='-', linewidth=1.8,
            label=r'$(4,1,1) = 137.0363$')

# Second best (3,7,-8)
ax2.axvline(SECOND_BEST, color='orange', linestyle=':', linewidth=1.8,
            label=r'$(3,7,-8) = 136.973$')

# Gap annotation
y_arrow = 0.85
ax2.annotate('', xy=(HIT_411, y_arrow), xytext=(SECOND_BEST, y_arrow),
             arrowprops=dict(arrowstyle='<->', color='black', lw=1.2))
ax2.text((HIT_411 + SECOND_BEST)/2, y_arrow + 0.04,
         f'gap {GAP_FACTOR:.0f}×',
         ha='center', fontsize=10, fontweight='bold')

ax2.set_xlabel(r'$a\pi^3 + b\pi^2 + c\pi$', fontsize=11)
ax2.set_ylabel('Count', fontsize=11)
ax2.set_title('(b) Zoom: region around the target', fontsize=12)
ax2.legend(fontsize=9, loc='upper left')
ax2.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('fig1_s1_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# ---------------------------------------------------------------
# Sanity check: print the key numbers
# ---------------------------------------------------------------
delta_411 = abs(HIT_411 - TARGET) / TARGET
delta_second = abs(SECOND_BEST - TARGET) / TARGET
print(f"(4,1,1)        = {HIT_411:.9f}   delta_rel = {delta_411:.3e}")
print(f"(3,7,-8)       = {SECOND_BEST:.9f}   delta_rel = {delta_second:.3e}")
print(f"Target         = {TARGET:.9f}")
print(f"Gap (2nd/1st)  = {delta_second/delta_411:.1f}x")
print()
print(f"N values in [{zoom_lo}, {zoom_hi}]: "
      f"{np.sum((values >= zoom_lo) & (values <= zoom_hi))}")

In [ ]:
import numpy as np
import itertools

# --- Target (CODATA 2018) ---
ALPHA_INV_EXP = 137.035999084
EPS_SCAN = 1e-3
K = 10

PI = np.pi
E  = np.e
S2 = np.sqrt(2)
PHI = (1 + np.sqrt(5)) / 2

SPACES = {
    "S1": [("pi^3", PI**3), ("pi^2", PI**2), ("pi", PI)],
    "S2": [("pi^3", PI**3), ("pi^2", PI**2), ("e", E), ("1", 1.0)],
    "S3": [("e^3", E**3), ("e^2", E**2), ("e", E)],
    "S4": [("pi^3", PI**3), ("pi^2", PI**2), ("sqrt2", S2), ("phi", PHI)],
    "S5": [("pi^4", PI**4), ("pi^3", PI**3), ("pi^2", PI**2), ("pi", PI)],
}


def sweep(basis):
    n = len(basis)
    values = [c for _, c in basis]
    hits = []
    for coeffs in itertools.product(range(-K, K+1), repeat=n):
        v = sum(a * c for a, c in zip(coeffs, values))
        err_abs = abs(v - ALPHA_INV_EXP)
        err_rel = err_abs / ALPHA_INV_EXP
        if err_rel < EPS_SCAN:
            hits.append((coeffs, v, err_abs, err_rel))
    hits.sort(key=lambda h: h[3])
    return hits


def latex_table(space_name, basis, hits):
    labels = [l for l, _ in basis]
    lines = []
    lines.append(r"\begin{table}[h]")
    lines.append(r"\centering")
    lines.append(r"\small")
    lines.append(
        r"\caption{All candidates in $\mathcal{%s}$ with $\delta_{\mathrm{rel}} < 10^{-3}$. "
        r"Basis: ${\%s}$. Total: %d candidates.}"
        % (space_name, ", ".join(labels), len(hits))
    )
    lines.append(r"\label{tab:appendix_%s}" % space_name.lower())
    lines.append(r"\begin{tabular}{r" + "r" * len(basis) + r"rr}")
    lines.append(r"\hline")
    header = r"Rank & " + " & ".join([f"$a_{{{i+1}}}$" for i in range(len(basis))]) + r" & Value & $\delta_{\mathrm{rel}}$ \\"
    lines.append(header)
    lines.append(r"\hline")
    for i, (coeffs, v, err_abs, err_rel) in enumerate(hits, start=1):
        coeff_str = " & ".join([f"{a:+d}" for a in coeffs])
        lines.append(
            f"{i} & {coeff_str} & {v:.9f} & {err_rel:.3e} \\"
        )
    lines.append(r"\hline")
    lines.append(r"\end{tabular}")
    lines.append(r"\end{table}")
    lines.append("")
    return "\n".join(lines)


def main():
    print(r"% Auto-generated by apendice_b.py")
    print(f"% Target: alpha^-1 = {ALPHA_INV_EXP:.12f} (CODATA 2018)")
    print(r"% Tolerance: eps_rel < 1e-3")
    print(r"% Bound: |a_i| <= %d" % K)
    print()

    summary_rows = []
    for name, basis in SPACES.items():
        hits = sweep(basis)
        n_1e6 = sum(1 for h in hits if h[3] < 1e-6)
        n_1e5 = sum(1 for h in hits if h[3] < 1e-5)
        n_1e3 = len(hits)
        summary_rows.append((name, n_1e6, n_1e5, n_1e3))
        print(latex_table(name, basis, hits))

    # Summary table
    print(r"\begin{table}[h]")
    print(r"\centering")
    print(r"\caption{Summary of hits per space and threshold.}")
    print(r"\label{tab:appendix_summary}")
    print(r"\begin{tabular}{lrrr}")
    print(r"\hline")
    print(r"Space & Hits @ $10^{-6}$ & Hits @ $10^{-5}$ & Hits @ $10^{-3}$ \\")
    print(r"\hline")
    for name, n6, n5, n3 in summary_rows:
        print(f"$\mathcal{{{name}}}$ & {n6} & {n5} & {n3} \\")
    print(r"\hline")
    print(r"\end{tabular}")
    print(r"\end{table}")


if __name__ == "__main__":
    main()

In [ ]:

# Anotar el gap entre (4,1,1) y (3,7,-8)
ax2.annotate('', xy=(137.0363, 0.9), xytext=(136.973, 0.9),
             arrowprops=dict(arrowstyle='<->', color='black'))
ax2.text(137.005, 0.95, 'gap 206×', ha='center', fontsize=10)

In [ ]:

ax2.axvline(136.973, color='orange', linestyle=':', linewidth=2,
            label=r'$(3,7,-8) = 136.973$')